In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'shared').exists():
    for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
        if (candidate / 'shared').exists() and (candidate / 'artifacts').exists():
            PROJECT_ROOT = candidate
            break

SRC_DIR = PROJECT_ROOT / 'artifacts' / 'ps-004-headcount-forecasting' / 'src'
MODELS_SRC_DIR = SRC_DIR / 'models'
for path in (SRC_DIR, MODELS_SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from arima_model import DATE_STAMP, train_arima_models
from model_evaluation import (
    FORECAST_FIGURE_PATH,
    FORECAST_TABLE_PATH,
    MODEL_COMPARISON_PATH,
    MODEL_REGISTRY_PATH,
    evaluate_models,
    generate_forecast,
    plot_forecast_by_profession,
    select_champions,
    write_contract_outputs,
 )

FEATURES_PATH = PROJECT_ROOT / 'artifacts' / 'ps-004-headcount-forecasting' / 'data' / '3_interim' / 'features.parquet'
MODELS_DIR = PROJECT_ROOT / 'artifacts' / 'ps-004-headcount-forecasting' / 'models'
HANDOFF_PATH = PROJECT_ROOT / 'docs' / 'agent-handoffs' / 'model-forecasting' / 'ps-004-headcount-forecasting' / 'forecasting_to_narrative_20260423.json'
PROFESSIONS = ['doctors', 'nurses', 'pharmacists', 'physiotherapists']

In [3]:
features = pl.read_parquet(FEATURES_PATH).sort(['profession', 'year'])
assert features.height == 48, f'Expected 48 feature rows, found {features.height}'
assert features.select(pl.col('year').max()).item() == 2019, 'Expected last observed year to be 2019'
assert features.filter(pl.col('profession') == 'physiotherapists').height == 6, 'Expected 6 physiotherapist rows'
print({
    'shape': features.shape,
    'year_range': (features.select(pl.col('year').min()).item(), features.select(pl.col('year').max()).item()),
})
features

{'shape': (48, 6), 'year_range': (2006, 2019)}


profession,year,count,year_index,lag_1,lag_2
cat,i32,i32,i32,i32,i32
"""doctors""",2006,6931,0,null,null
"""doctors""",2007,7464,1,6931,null
"""doctors""",2008,7841,2,7464,6931
"""doctors""",2009,8323,3,7841,7464
"""doctors""",2010,9030,4,8323,7841
…,…,…,…,…,…
"""physiotherapists""",2015,1549,1,1394,null
"""physiotherapists""",2016,1693,2,1549,1394
"""physiotherapists""",2017,1814,3,1693,1549


In [4]:
arima_models = train_arima_models(features, holdout_years=3)
arima_summary = pl.DataFrame([
    {
        'profession': profession,
        'selected_order': None if order is None else str(order),
        'aic': None if not hasattr(model, 'aic') else round(float(model.aic), 4),
    }
    for profession, (model, order) in arima_models.items()
]).sort('profession')
arima_summary

WARNING [doctors] ARIMA(1, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [doctors] ARIMA(2, 0, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [doctors] ARIMA(2, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
INFO [doctors] selected ARIMA(1, 1, 2) with AIC=131.077
WARNING [nurses] ARIMA(2, 1, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
INFO [nurses] selected ARIMA(1, 1, 2) with AIC=160.045
WARNING [pharmacists] ARIMA(2, 0, 1) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [pharmacists] ARIMA(2, 0, 2) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [pharmacists] ARIMA(2, 1, 1) convergence warning: Maximum Likelihood optimization failed to converge. Check mle_retvals
WARNING [p

profession,selected_order,aic
str,str,f64
"""doctors""","""(1, 1, 2)""",131.0775
"""nurses""","""(1, 1, 2)""",160.0449
"""pharmacists""","""(1, 1, 0)""",108.6623
"""physiotherapists""","""(1, 1, 0)""",24.0992


In [5]:
linear_models = {
    profession: joblib.load(MODELS_DIR / f'{profession}_linear_{DATE_STAMP}.pkl')
    for profession in PROFESSIONS
}
eval_df = evaluate_models(features, linear_models, arima_models, holdout_years=3)
model_comparison = eval_df.select(['profession', 'model_type', 'mape', 'mae', 'rmse']).sort(['profession', 'model_type'])
model_comparison

profession,model_type,mape,mae,rmse
str,str,f64,f64,f64
"""doctors""","""arima""",1.5406,215.4938,244.6269
"""doctors""","""linear""",2.2538,314.7303,346.8399
"""nurses""","""arima""",2.9819,1262.6283,1373.8763
"""nurses""","""linear""",8.6084,3639.6727,3817.8366
"""pharmacists""","""arima""",4.1095,135.1503,149.9334
"""pharmacists""","""linear""",2.0482,66.7848,69.8195
"""physiotherapists""","""arima""",2.326,46.0152,61.122
"""physiotherapists""","""linear""",3.0473,60.1667,74.9472


In [6]:
champion_df = select_champions(eval_df)
champion_models = {}
for row in champion_df.to_dicts():
    profession = row['profession']
    if row['model_type'] == 'linear':
        champion_models[profession] = {'model_type': 'linear', 'model': linear_models[profession]}
    else:
        champion_models[profession] = {'model_type': 'arima', 'model': arima_models[profession][0]}
champion_df

profession,model_type,mape,mae,rmse,mape_target_met
str,str,f64,f64,f64,bool
"""doctors""","""arima""",1.5406,215.4938,244.6269,true
"""nurses""","""arima""",2.9819,1262.6283,1373.8763,true
"""pharmacists""","""linear""",2.0482,66.7848,69.8195,true
"""physiotherapists""","""arima""",2.326,46.0152,61.122,true


In [7]:
forecast_df = generate_forecast(features, champion_models, horizon=5)
forecast_table = forecast_df.select(['profession', 'year', 'forecast_count', 'model_type', 'lower_95', 'upper_95'])
contract_paths = write_contract_outputs(eval_df, champion_df, forecast_df)
plot_forecast_by_profession(features, forecast_df, FORECAST_FIGURE_PATH)
HANDOFF_PATH.parent.mkdir(parents=True, exist_ok=True)
handoff_payload = {
    'status': 'success',
    'agent': 'model-forecasting',
    'problem_statement': 'ps-004-headcount-forecasting',
    'timestamp': '20260423',
    'outputs': {
        'arima_script': 'artifacts/ps-004-headcount-forecasting/src/models/arima_model.py',
        'evaluation_script': 'artifacts/ps-004-headcount-forecasting/src/model_evaluation.py',
        'notebook': 'artifacts/ps-004-headcount-forecasting/notebooks/03_modeling_and_forecast.ipynb',
        'model_comparison_csv': 'artifacts/ps-004-headcount-forecasting/results/metrics/model_comparison.csv',
        'model_registry_csv': 'artifacts/ps-004-headcount-forecasting/results/metrics/model_registry.csv',
        'forecast_table_csv': 'artifacts/ps-004-headcount-forecasting/results/exports/forecast_table.csv',
        'forecast_figure': 'artifacts/ps-004-headcount-forecasting/reports/figures/forecast_by_profession.png'
    },
    'champion_models': {row['profession']: row['model_type'] for row in champion_df.to_dicts()},
    'forecast_years': sorted(forecast_df.get_column('year').unique().to_list()),
    'mape_target_met': {row['profession']: row['mape_target_met'] for row in champion_df.to_dicts()}
}
HANDOFF_PATH.write_text(json.dumps(handoff_payload, indent=2))
forecast_table

profession,year,forecast_count,model_type,lower_95,upper_95
str,i64,f64,str,f64,f64
"""doctors""",2020,13442.823,"""arima""",13253.3873,13632.2587
"""doctors""",2021,14025.8322,"""arima""",13603.7277,14447.9368
"""doctors""",2022,14608.8263,"""arima""",14024.1442,15193.5083
"""doctors""",2023,15191.8052,"""arima""",14465.8855,15917.7248
"""doctors""",2024,15774.769,"""arima""",14918.3101,16631.2279
…,…,…,…,…,…
"""physiotherapists""",2020,1836.604,"""arima""",1815.0706,1858.1374
"""physiotherapists""",2021,1979.8131,"""arima""",1931.7158,2027.9104
"""physiotherapists""",2022,2122.6284,"""arima""",2042.2315,2203.0252


In [8]:
print('Contract files written:', [str(path) for path in [*contract_paths, FORECAST_FIGURE_PATH, HANDOFF_PATH]])

Contract files written: ['/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/results/metrics/model_comparison.csv', '/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/results/metrics/model_registry.csv', '/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/results/exports/forecast_table.csv', '/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/artifacts/ps-004-headcount-forecasting/reports/figures/forecast_by_profession.png', '/Users/alfredtang/Documents/Projects/gen-e2/gen-e2-data-analysis/docs/agent-handoffs/model-forecasting/ps-004-headcount-forecasting/forecasting_to_narrative_20260423.json']
